# SSL, Fine Tuning, and Linear Probing Heads

> none of these work! 

In [ ]:
#| default_exp heads

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn.functional as F

from torch import nn
import math
from torch.nn import init
from wavefm.patchtst import TSTBlock
from wavefm.layers import InceptionBlock, Transpose, get_activation_fn, EmbeddingAugmentations, MultiheadAttentionCustom, MultiHeadAttention, PositionalEncoding, LearnableMaskedChannelTokens
from wavefm.utils import trunc_normal_
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.nn.attention import SDPBackend, sdpa_kernel
from wavefm.jepa import MLP, JEPABlock
from wavefm.nested import unflatten_dim_from_batch, flatten_dim_to_batch

## Linear Probing and Fine Tuning Heads

In [ ]:
#| export
class RNNProbingHead(nn.Module):
    def __init__(self, 
                 c_in, 
                 input_size, 
                 hidden_size, 
                 n_classes,
                 missing_channel_indices=None,
                 module='GRU', 
                 rnn_dropout=0., 
                 num_rnn_layers=1,
                 pool='average', # 'average' or 'max' or 'majority'
                 predict_every_n_patches=1,
                 bidirectional=True, 
                 affine=False, 
                 pre_norm=True,
                 mlp_final_head=False,
                 linear_dropout=0.
                 ):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.c_in = c_in
        self.rnn_dropout = rnn_dropout
        self.bidirectional = bidirectional
        self.n_classes = n_classes
        self.pool = pool
        self.num_rnn_layers = num_rnn_layers
        self.predict_every_n_patches = predict_every_n_patches
        self.affine = affine
        self.missing_channel_indices = missing_channel_indices


        # if self.missing_channel_indices is not None:
        #     # expects : [bs x nvars x d_model x num_patch] 
        #     self.masked_channel_token_layer = LearnableMaskedChannelTokens(missing_channel_indices=self.missing_channel_indices, d_model=self.input_size) 
        # else:
        #     self.masked_channel_token_layer = nn.Identity()

        if pre_norm:
            self.norm_layer = nn.LayerNorm(self.input_size)
        else:
            self.norm_layer = nn.Identity()

        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(c_in))
            self.affine_bias = nn.Parameter(torch.zeros(c_in))

        self.flatten = nn.Flatten(start_dim=-2, end_dim=-1) # flatten d_model and n channels

        if module.lower() == 'gru':
            self.rnn = nn.GRU(input_size=self.input_size * self.c_in, hidden_size=self.hidden_size, num_layers=num_rnn_layers, bias=True, batch_first=True, dropout=rnn_dropout, bidirectional=self.bidirectional)
        else:
            self.rnn = nn.LSTM(input_size=self.input_size * self.c_in, hidden_size=self.hidden_size, num_layers=num_rnn_layers, bias=True, batch_first=True, dropout=rnn_dropout, bidirectional=self.bidirectional)

        f = 2 if self.bidirectional else 1

        if mlp_final_head:
            self.linear_head = nn.Sequential(
                nn.Linear(hidden_size*f, hidden_size*f//2),
                nn.LayerNorm([hidden_size*f//2]),
                nn.ReLU(),
                nn.Dropout(linear_dropout),
                nn.Linear(hidden_size*f//2, n_classes)
            )
        else:
            self.linear_head = nn.Linear(hidden_size*f, n_classes) 
        
        if not self.predict_every_n_patches:
            self.pool_layer = nn.AdaptiveAvgPool1d(1)
        else:
            self.pool_kernel_size = int(self.predict_every_n_patches)
            self.pool_layer = nn.AvgPool1d(kernel_size=self.pool_kernel_size, stride=self.pool_kernel_size) if self.pool != 'max' else nn.MaxPool1d(kernel_size=self.pool_kernel_size, stride=self.pool_kernel_size)

        self.softmax = nn.Softmax(dim=1)
        
    def forward(self, x, return_softmax=False, channel_mask=None):
        """
        GRU input expects [bs x seq_len x H_in]
        x: new: [bs * c_in x n patch x d_model] previous: [bs x n channels x d model x n patches]
        """
        # pass each channel through GRU and concat
        bs = x.size(0) // self.c_in
        was_nested = x.is_nested
        if was_nested:
            x = unflatten_dim_from_batch(x, self.c_in) # [bs x n channels x n patches x d model] 
            lengths = [i.size(1) for i in x.unbind()]
            x = x.to_padded_tensor(padding=0) # not output size specified
        else:
            lengths = None
            x = x.reshape(bs, self.c_in, -1, self.input_size) # [bs x n channels x n patches x d model]
        #x = self.masked_channel_token_layer(x) # [bs x nvars x d_model x num_patch] 
        #x = x.transpose(2,3) # [bs x n channels x n patches x d model]
        x = self.norm_layer(x) # [bs x n channels x n patches x d model]
        x = x.permute(0,2,3,1) # [bs x n patches x d model x n channels]
        if self.affine:
             x = x * self.affine_weight
             x = x + self.affine_bias
        #x = x.mean(dim=-1) # [bs x n patches x d model]
        x = self.flatten(x) # [bs x n patches x n channels * d model]
        #x = x.transpose(1,2) # [bs x n channels * d model x n patches]
        if was_nested:
            x = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        x,hidden = self.rnn(x) # [bs x n patches x hidden_size]

        if was_nested:
            # note that we cannot convert back to a nested tensor because of backprop errors through RNN
            # however, if the batch max length is short, all in the batch will be short, so it is still necessary to pad at some point

            x, _ = pad_packed_sequence(x, batch_first=True, padding_value=0, total_length=max(lengths))
            #x_list = [x[i, :length, :].contiguous() for i, length in enumerate(lengths)]
            #x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
        x = self.linear_head(x) # [bs x n patches x n classes]
        x = x.transpose(1,2)
        # if x.is_nested:
        #     x_list = list(x.unbind())
        #     for i, x_i in enumerate(x_list):
        #         t = self.pool_layer(x_i)
        #         t = t.transpose(0,1) # to ensure correct jagged dim is assigned
        #         x_list[i] = t
        #     x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
        #     x = x.transpose(1,2).to_padded_tensor(padding=0)
        # else:
        x = self.pool_layer(x)
        if return_softmax:
            x = self.softmax(x)
        return x
     


In [ ]:
#| notest
m = RNNProbingHead(c_in=7, 
                    pool='average', 
                    input_size = 384, 
                    missing_channel_indices=None,#,[0,1],
                    bidirectional=True,
                    affine=True, 
                    hidden_size=1200,
                    module='GRU',
                    n_classes=5,
                    predict_every_n_patches=5,
                    rnn_dropout=0.,
                    num_rnn_layers=1,
                    mlp_final_head=True,
                    pre_norm=True)
x = torch.randn((4*7,960,384))
m(x, return_softmax=True).shape


torch.Size([4, 5, 192])

In [ ]:
#| notest
batch_size = 2
n_vars = 7
max_len = 480
d_model = 384

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, length, d_model) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
x_nested = flatten_dim_to_batch(x_nested, dim=1)  # flatten n_vars into batch dimension
print(x_nested.shape)
y = m(x_nested, return_softmax=False)
y.shape
# #from wavefm.loss import mse_loss

# #y = y.mean(dim=1)
# print(y.shape)
# mse_loss(y, torch.ones_like(y)).backward()

torch.Size([14, j2, 384])


NameError: name 'm' is not defined

In [ ]:
#| export
class SingleAttentionLayer(nn.Module):
    def __init__(self, hidden_size, attention_dim=512):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, attention_dim),
            nn.Tanh(),
            nn.Linear(attention_dim, 1)
        )
        
    def forward(self, encoder_outputs, mask=None):
        """
        Args:
            encoder_outputs: [batch_size, seq_len, hidden_size]
            mask: [batch_size, seq_len] optional attention mask
        Returns:
            attention_weights: [batch_size, seq_len]
            weighted_output: [batch_size, hidden_size]
        """
        scores = self.attention(encoder_outputs).squeeze(-1)  # [batch_size, seq_len]
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
            
        attention_weights = F.softmax(scores, dim=-1)  # [batch_size, seq_len]
        weighted_output = torch.bmm(attention_weights.unsqueeze(1), 
                                  encoder_outputs).squeeze(1)  # [batch_size, hidden_size]
        
        return attention_weights, weighted_output

class AttentionRNNProbingHead(nn.Module):
    def __init__(self, 
                 c_in, 
                 input_size, 
                 hidden_size, 
                 n_classes, 
                 attention_dim=512,
                 module='GRU', 
                 rnn_dropout=0., 
                 num_rnn_layers=1, 
                 predict_every_n_patches=1, 
                 bidirectional=True, 
                 affine=False, 
                 shared_embedding=True,
                 augmentations=None,
                 augmentation_mask_ratio=0.,
                 augmentation_dims_to_shuffle=[1,2,3],
                 pool='average', # 'average' or 'max'
                 norm=None, # one of [None, 'pre', 'post']
                 attn_type='single', # 'multi' or 'single'
                 n_heads=2 # only used if attn_type='multi'
                 ):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.c_in = c_in
        self.rnn_dropout = rnn_dropout
        self.bidirectional = bidirectional
        self.n_classes = n_classes
        self.num_rnn_layers = num_rnn_layers
        self.predict_every_n_patches = predict_every_n_patches
        self.affine = affine
        self.shared_embedding = shared_embedding
        self.augmentations = augmentations
        self.pool = pool
        self.attn_type = attn_type
        self.n_heads = n_heads
        self.attention_dim = attention_dim
        if augmentations is not None:
            self.augmentations = augmentations
            self.augmentation_layer = EmbeddingAugmentations(augmentations=augmentations, dims_to_shuffle=augmentation_dims_to_shuffle, patch_mask_ratio=augmentation_mask_ratio, jitter_zero_mask_ratio=augmentation_mask_ratio)
        if norm in ['pre', 'post']:
            self.do_norm = norm
            self.norm_layer = nn.BatchNorm2d(self.c_in)# nn.LayerNorm(self.hidden_size * (2 if self.bidirectional else 1))
        else:
            self.do_norm = False
            self.norm_layer = nn.Identity()
        self.flatten = nn.Flatten(start_dim=-3, end_dim=-2)
        
        if attn_type == 'single':
            self.attn = SingleAttentionLayer(hidden_size=attention_dim, attention_dim=hidden_size)
        elif attn_type == 'multi':
            self.attn = MultiheadAttention(d_model=attention_dim, n_heads=n_heads)
        else:
            self.attn = nn.Identity()

        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(1,c_in,1))
            self.affine_bias = nn.Parameter(torch.zeros(1,c_in,1))

        if module.lower() == 'gru':
            self.rnn = nn.GRU(input_size=self.input_size+attention_dim, hidden_size=self.hidden_size, num_layers=num_rnn_layers, bias=True, batch_first=True, dropout=rnn_dropout, bidirectional=self.bidirectional)
        else:
            self.rnn = nn.LSTM(input_size=self.input_size+attention_dim, hidden_size=self.hidden_size, num_layers=num_rnn_layers, bias=True, batch_first=True, dropout=rnn_dropout, bidirectional=self.bidirectional)
        
        self.layers = []
        f = 2 if self.bidirectional else 1
        if not self.shared_embedding:
            for _ in range(self.c_in): self.layers.append(nn.Linear(hidden_size*f, n_classes))
        else:
            self.layers.append(nn.Linear(hidden_size*f, n_classes)) # [bs x c_in x n_patches x n_classes]
        self.layers = nn.Sequential(*self.layers)
        self.avg_pool_kernel_stride = int(self.c_in*self.predict_every_n_patches) if self.predict_every_n_patches > 1 else self.c_in
        self.avg_pool = nn.AvgPool1d(kernel_size=self.avg_pool_kernel_stride, stride=self.avg_pool_kernel_stride) if self.pool == 'average' else nn.MaxPool1d(kernel_size=self.avg_pool_kernel_stride, stride=self.avg_pool_kernel_stride)

        self.softmax = nn.LogSoftmax(dim=1)
        
    def forward(self, x, return_softmax=False):
        """
        GRU input expects [bs x seq_len x H_in]
        x: [bs x n channels x d model x n patches]
        """
        bs, n_channels, d_model, n_patches = x.shape
        if self.augmentations is not None:
            x = self.augmentation_layer(x)
        if self.do_norm == 'pre':
            x = self.norm_layer(x)
        x_cat = x.permute(0, 3, 1, 2) # [bs x n patches x n channels x d model]
        x_cat = self.flatten(x_cat) # [bs x n patches*n channels x d model]
        if self.attn_type == 'single':
            attn_weights, context = self.attn(x_cat) # [bs x n patches*n channels x hidden_size]
            context = context.unsqueeze(1).expand(-1, n_patches*n_channels, -1)
            # Concatenate encoder outputs with context vector
            x_cat = torch.cat([x_cat, context], dim=-1)
        else:
            context = self.attn(x_cat)
            x_cat = torch.cat([x_cat, context], dim=-1)
        x_cat,_ = self.rnn(x_cat)
        x_cat = torch.reshape(x_cat, (bs, n_patches, n_channels, -1))
        if self.do_norm == 'post':
            x_cat = x_cat.permute(0, 2, 3, 1)  # [bs, n_channels, hidden_size, n_patches]
            x_cat = self.norm_layer(x_cat)
            x_cat = x_cat.permute(0, 3, 1, 2)  # [bs, n_patches, n_channels, hidden_size]
            
        if not self.shared_embedding:
            x_out = []
            for i in range(self.c_in):
                z_ = self.layers[i](x_cat[:,:,i,:])
                x_out.append(z_)
            x_cat = torch.stack(x_out, dim=2) # [bs x n patches x n channels x n classes]
        else:
            x_cat = self.layers(x_cat) # [bs x n patches x n channels x n classes]
        
        if self.affine:
            x_cat = x_cat * self.affine_weight
            x_cat = x_cat + self.affine_bias
        x_cat = self.flatten(x_cat)
        x_cat = x_cat.transpose(1,2)
        x_cat = self.avg_pool(x_cat)
        if return_softmax:
            x_cat = self.softmax(x_cat)
        return x_cat

In [ ]:
#| notest
m = AttentionRNNProbingHead(c_in=7, input_size = 512, bidirectional=True, affine=True, shared_embedding=False, hidden_size=384, module='GRU', n_classes=2, predict_every_n_patches=960, rnn_dropout=0., num_rnn_layers=1, norm='post', attn_type='multi')
x = torch.randn((4,7,512,960))

m(x, return_softmax=True).shape

NameError: name 'MultiheadAttention' is not defined

In [ ]:
#| export
class TransformerDecoderProbingHead(nn.Module):
    def __init__(self, 
                 c_in, 
                 d_model, 
                 n_classes, 
                 norm='BatchNorm', 
                 dropout=0., 
                 act='gelu', 
                 d_ff=2048, 
                 num_layers=1, 
                 n_heads=2, 
                 predict_every_n_patches=1, 
                 affine=False, 
                 shared_embedding=True
                 ):


        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.num_layers = num_layers
        self.d_ff = d_ff
        self.act = act
        self.c_in = c_in
        self.dropout = dropout
        self.n_classes = n_classes
        self.predict_every_n_patches = predict_every_n_patches
        self.affine = affine
        self.shared_embedding = shared_embedding

        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(1,c_in,1))
            self.affine_bias = nn.Parameter(torch.zeros(1,c_in,1))

        self.flatten = nn.Flatten(start_dim=-3, end_dim=-2)
        norm_m = nn.BatchNorm1d(d_model) if "batch" in norm.lower() else nn.LayerNorm([d_model])
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=n_heads, dropout=dropout, dim_feedforward=d_ff, batch_first = True, activation=act)
        self.decoder =  nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        self.layers = []
        if not self.shared_embedding:
            for _ in range(self.c_in): self.layers.append(nn.Linear(d_model, n_classes))
        else:
            self.layers.append(nn.Linear(d_model, n_classes)) # [bs x c_in x n_patches x n_classes]
        self.layers = nn.Sequential(*self.layers)

        avg_pool_kernel_stride = int(self.c_in*self.predict_every_n_patches) if self.predict_every_n_patches > 1 else self.c_in
        self.avg_pool = nn.AvgPool1d(kernel_size=avg_pool_kernel_stride, stride=avg_pool_kernel_stride)
        self.softmax = nn.LogSoftmax(dim=1)
        
    def forward(self, x, return_softmax=False):
        """
        x: [bs x n channels x d model x n patches]
        """
        # pass each channel through GRU and concat
        bs, n_channels, d_model, n_patches = x.shape
        x = x.permute(0,1,3,2) #[bs x n channels x n patches x d model]
        x = x.reshape(-1, n_patches, d_model)
        x = self.decoder(x,x) # x is both target and memory
        x = x.reshape(bs, n_channels, n_patches, d_model).permute(0,2,1,3) # with just rnn only need permute(0, 3, 1, 2) [bs x n patches x n channels x d model]

        if not self.shared_embedding:
            x_out = []
            for i in range(self.c_in):
                z_ = self.layers[i](x[:,:,i,:])
                x_out.append(z_)
            x = torch.stack(x_out, dim=2) # [bs x n patches x n channels x n classes]
        else:
            x = self.layers(x) # [bs x n patches x n channels x n classes]
        
        if self.affine:
            x = x * self.affine_weight
            x = x + self.affine_bias

        x = self.flatten(x)
        x = x.transpose(1,2)
        x = self.avg_pool(x)
        #x = x.transpose(1,2)
        #x = x.reshape(x.shape[0], self.c_in, -1, self.n_classes)
        if return_softmax:
            x = self.softmax(x)
        return x

In [ ]:
#| notest
layer = TransformerDecoderProbingHead(c_in=7, affine=True, norm='LayerNorm', shared_embedding=False, d_model=512, n_classes=2, dropout=0., num_layers=1, n_heads=2, predict_every_n_patches=3600)
x = torch.randn((4, 7, 512, 3600))

layer(x).shape

torch.Size([4, 2, 1])

In [ ]:
#| export
class DecoderFeedForward(nn.Module):
    """
    transformer decoder with attention for feedforward predictions. This is really just another encoder layer followed by
    a linear layer + 1d convolution + softmax. However, if used in linear probing, could be useful.
    """
    def __init__(self, 
                 c_in, # the number of input channels
                 predict_every_n_patches, # for a given sequence of length m with frequency f, number of predictions
                 num_layers,
                 d_ff,
                 attn_dropout,
                 res_attention, 
                 pre_norm, 
                 store_attn,
                 n_heads,
                 shared_embedding,
                 affine,
                 n_classes, # the number of classes to predict (for sleep stage - there are 6)
                 d_model, # the dimension of the transformer model
                 norm='BatchNorm', # batchnorm or layernorm between linear and convolutional layers
                 act='gelu', # activation function to use between layers, 'gelu' or 'relu'
                 dropout=0. # dropout in between linear layers
                 ):
        super().__init__()
        assert n_classes >= 2, "The number of classes must be >= 2. If you want to perform a binary task, pass in the value 2."
        self.c_in = c_in
        self.n_classes = n_classes
        self.predict_every_n_patches = predict_every_n_patches
        self.d_model = d_model
        self.num_layers = num_layers
        self.d_ff = d_ff
        self.attn_dropout = attn_dropout
        self.res_attention = res_attention
        self.pre_norm = pre_norm
        self.store_attn = store_attn
        self.n_heads = n_heads
        self.norm = norm
        self.act = act
        self.dropout = dropout
        self.shared_embedding = shared_embedding
        self.affine = affine

        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(1,c_in,1))
            self.affine_bias = nn.Parameter(torch.zeros(1,c_in,1))
        
        self.decoder_layers = nn.ModuleList([TSTBlock(d_model=d_model, n_heads=n_heads, d_ff=d_ff, norm=norm, use_flash_attn=True,
                                                      attn_dropout=attn_dropout, dropout=dropout,
                                                      activation=act, res_attention=res_attention,
                                                      pre_norm=pre_norm, store_attn=store_attn) for i in range(num_layers)])
        self.layers = []
        if not self.shared_embedding:
            for _ in range(self.c_in): self.layers.append(nn.Linear(d_model, n_classes))
        else:
            self.layers.append(nn.Linear(d_model, n_classes)) # [bs x c_in x n_patches x n_classes]
        self.layers = nn.Sequential(*self.layers)
        
        #self.linear_layer = nn.Sequential(nn.Linear(d_model, self.win_length), nn.BatchNorm2d(c_in) if "batch" in norm.lower() else nn.LayerNorm([c_in, n_patches, self.win_length]), get_activation_fn(act), nn.Dropout(dropout))
        #self.flatten = nn.Flatten(start_dim=-2) # now flatten everything to get full length sequences
        # now convolve with stride/kernel = frequency * n second predicitions
        #final_kernel_stride = int(frequency*predict_every_seconds) # this will make one predictions for every predict_every_seconds seq length
        #self.conv1d = nn.Conv1d(in_channels=c_in, out_channels=n_classes, kernel_size=final_kernel_stride, stride=final_kernel_stride)
        #self.unfold_size = int(self.predict_every_seconds/(self.win_length/self.frequency))
        self.flatten = nn.Flatten(start_dim=-3, end_dim=-2)
        self.avg_pool_kernel_stride = int(self.c_in*self.predict_every_n_patches) if self.predict_every_n_patches > 1 else self.c_in
        self.avg_pool = nn.AvgPool1d(kernel_size=self.avg_pool_kernel_stride, stride=self.avg_pool_kernel_stride)
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x, key_padding_mask=None, return_softmax=False):
        """
        Make sure to freeze the encoder and remove its pretrain head! 

        in: [bs x n channels x d model x n patches]
        out: [bs x n classes x pred len]
        """
        bs, n_channels, d_model, n_patches = x.shape
        z = x.permute(0,1,3,2) #[bs x n channels x n patches x d model]
        z = z.reshape(-1, n_patches, d_model) # z: [bs * nvars x num_patch x d_model]
        if self.res_attention:
            scores = None
            for mod in self.decoder_layers:
                z, scores = mod(z, prev=scores, key_padding_mask=key_padding_mask) # z: [bs * n_vars x num_patch x d_model], scores: [bs * n_vars x n_heads x num_patch x num_patch]
        else:
            for mod in self.decoder_layers: 
                z = mod(z, key_padding_mask=key_padding_mask) # z: [bs * n_vars x num_patch x d_model]
        z = z.reshape(bs, n_channels, n_patches, d_model).permute(0,2,1,3) # z: [bs x n patches x n channels x d model]
        if not self.shared_embedding:
            x_out = []
            for i in range(self.c_in):
                z_ = self.layers[i](z[:,:,i,:])
                x_out.append(z_)
            z = torch.stack(x_out, dim=2) # [bs x n patches x n channels x n classes]
        else:
            z = self.layers(z) # [bs x n patches x n channels x n classes]

        if self.affine:
            z = z * self.affine_weight
            z = z + self.affine_bias
        z = self.flatten(z) # # [bs x n channels * n patches x n classes]
        z = z.transpose(1,2) # [bs x n classes x n channels * n patches]
        z  = self.avg_pool(z)
        #z = z.unfold(-2, size=self.unfold_size, step=self.unfold_size).sum(dim=-1) # take 30 seconds and sum up logits [bs x n channels x 30sec patches x n_classes]
        
        #z = self.flatten(z)
        #z=z.sum(dim=1) 
        #z = z.transpose(1,2)
        #z = self.conv1d(z)
        if return_softmax:
            z = self.softmax(z)
        return z

In [ ]:
#| notest
c_in = 7
frequency = 125
win_length=750 
overlap = 0.
hop_length=win_length - int(overlap*win_length)
max_seq_len_sec = (6*3600) # for dataloader
#seq_len_sec = sample_stride = 3*3600 # for dataloader
max_seq_len = max_seq_len_sec*frequency # for model
#n_patches = n_fft // 2 + 1
n_patches = (max(max_seq_len, win_length)-win_length) // hop_length + 1

#patch_len = int((win_length-conv_kernel_stride_size[1])/conv_kernel_stride_size[1] + 1)
x = torch.randn(2,c_in,512,n_patches)

model = DecoderFeedForward(c_in=c_in,
                           predict_every_n_patches=5,
                           num_layers=1,
                           d_ff = 2048,
                           attn_dropout=0.,
                           res_attention = False,
                           pre_norm = False,
                           store_attn = False,
                           n_heads=2,
                           affine=False,
                           shared_embedding=False,
                           n_classes=5,
                           d_model=512,
                           norm='BatchNorm',
                           act='gelu',
                           dropout=0.
                           )

model(x).shape

torch.Size([2, 5, 720])

In [ ]:
#| export
class ConvolutionalClassifier(nn.Module):
    """
    """
    def __init__(self, 
                 c_in, # the number of input channels
                 frequency, # the frequency of the original channels
                 predict_every_seconds, # for a given sequence of length m with frequency f, number of predictions
                 n_classes, # the number of classes to predict (for sleep stage - there are 6)
                 win_length, # the convolved patch length, the first step in this is to do a linear layer to this dimension
                 d_model, # the dimension of the transformer model
                 affine = False,
                 shared_embedding=True
                 ):
        super().__init__()
        
        self.c_in = c_in
        self.n_classes = n_classes
        self.predict_every_seconds = predict_every_seconds
        self.d_model = d_model
        self.win_length = win_length
        self.frequency = frequency
        self.layers = []
        self.affine = affine
        self.shared_embedding = shared_embedding
        self.convs = nn.ModuleList([nn.Conv2d(in_channels=1, out_channels=num_filters, kernel_size=size) for size in filter_sizes])

        if not self.shared_embedding:
            for _ in range(self.c_in): self.layers.append(nn.Linear(d_model, win_length))
        else:
            self.layers.append(nn.Linear(d_model, win_length)) # [bs x c_in x n_patches x n_classes]

        self.layers = nn.Sequential(*self.layers)
        # now we should probably to a 2d convtranspose to get to the original number of channels, now the channels and patch len are back to normal size
        self.flatten = nn.Flatten(start_dim=-2) # now flatten everything to get full length sequences
        # now convolve with stride/kernel = frequency * n second predicitions
        final_kernel_stride = int(frequency*predict_every_seconds) # this will make one predictions for every predict_every_seconds seq length
        self.conv1d = nn.Conv1d(in_channels=c_in, out_channels=n_classes, kernel_size=final_kernel_stride, stride=final_kernel_stride)
        self.softmax = nn.LogSoftmax(dim=1)
    
    def forward(self, x, return_softmax=False):
        """
        Make sure to freeze the encoder and remove its pretrain head! 

        in: [bs x n channels x d model x n patches]
        out: [bs x n classes x pred len]
        """
        x = x.permute(0,3,1,2) # [bs x n patches x n channels x d model]
        if not self.shared_embedding:
            x_out = []
            for i in range(self.c_in):
                z_ = self.layers[i](x[:,:,i,:])
                x_out.append(z_)
            x = torch.stack(x_out, dim=2) # [bs x n patches x n channels x patch len]
        else:
            x = self.layers(x) # [bs x n patches x n channels x patch len]
        if self.affine:
            x = x * self.affine_weight
            x = x + self.affine_bias
        x = x.transpose(1,2) # [bs x n channels x n patches x patch len]
        x = self.flatten(x)
        x = self.conv1d(x)
        if return_softmax:
            x = self.softmax(x)
        return x

In [ ]:
#| export
class TimeDistributedConvolutionalFeedForward(nn.Module):
    """
    Convolutional feed forward head that first uses a linear feed forward network to project features into the original convolutional dimension.
    Then, a convolutional transpose is used to extrapolate the data to its original form.
    Finally, a final convolution is used to predict the classes.
    """
    def __init__(self, 
                 c_in, # the number of input channels
                 kernel_size, # for a given sequence of length m with frequency f, number of predictions
                 n_classes, # the number of classes to predict (for sleep stage - there are 6)
                 d_model, # the dimension of the transformer model
                 affine = False,
                 dropout=0.,
                 shared_embedding=True
                 ):
        super().__init__()
        
        self.c_in = c_in
        self.n_classes = n_classes
        self.kernel_size = kernel_size
        self.d_model = d_model
        self.layers = []
        self.affine = affine
        self.shared_embedding = shared_embedding
        
        self.pool = nn.AdaptiveAvgPool1d(1)
        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(1,c_in,1))
            self.affine_bias = nn.Parameter(torch.zeros(1,c_in,1))

        if not self.shared_embedding:
            for _ in range(self.c_in): self.layers.append(nn.Linear(d_model, d_model // 2))
        else:
            self.layers.append(nn.Linear(d_model, d_model // 2)) # [bs x c_in x n_patches x n_classes]

        self.layers = nn.Sequential(*self.layers)
        self.norm = nn.LayerNorm([d_model // 2])
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        # now we should probably to a 2d convtranspose to get to the original number of channels, now the channels and patch len are back to normal size
        self.flatten = nn.Flatten(start_dim=-2) # now flatten everything to get full length sequences
        # now convolve with stride/kernel = frequency * n second predicitions
        self.conv1d = nn.Conv1d(in_channels=c_in, out_channels=n_classes, kernel_size=kernel_size, stride=1)
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x, return_softmax=False):
        """
        Make sure to freeze the encoder and remove its pretrain head! 

        in: [bs x n channels x d model x n patches]
        out: [bs x n classes x pred len]
        """
        x = x.permute(0,3,1,2) # [bs x n patches x n channels x d model]
        if not self.shared_embedding:
            x_out = []
            for i in range(self.c_in):
                z_ = self.layers[i](x[:,:,i,:])
                x_out.append(z_)
            x = torch.stack(x_out, dim=2) # [bs x n patches x n channels x d_model // 2]
        else:
            x = self.layers(x) # [bs x n patches x n channels x d_model // 2]
        if self.affine:
            x = x * self.affine_weight
            x = x + self.affine_bias
        x = self.norm(x)
        x = self.gelu(x)
        x = self.dropout(x)
        x = x.transpose(1,2) # [bs x n channels x n patches x d_model // 2]
        x = self.flatten(x)
        x = self.conv1d(x)
        x = self.pool(x)
        if return_softmax:
            x = self.softmax(x)
        return x

In [ ]:
#| notest
c_in = 7
frequency = 125
win_length=750 
overlap = 0.
hop_length=win_length - int(overlap*win_length)
max_seq_len_sec = (8*3600) # for dataloader
#seq_len_sec = sample_stride = 3*3600 # for dataloader
max_seq_len = max_seq_len_sec*frequency # for model
#n_patches = n_fft // 2 + 1
n_patches = (max(max_seq_len, win_length)-win_length) // hop_length + 1

#patch_len = int((win_length-conv_kernel_stride_size[1])/conv_kernel_stride_size[1] + 1)
x = torch.randn(4,7,512,n_patches)

model = TimeDistributedConvolutionalFeedForward(c_in=7, 
                                                kernel_size=20, 
                                                n_classes=2, 
                                                d_model=512,
                                                shared_embedding=True,
                                                affine=True
                                                )

print(model)
model(x).shape

TimeDistributedConvolutionalFeedForward(
  (pool): AdaptiveAvgPool1d(output_size=1)
  (layers): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
  )
  (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (gelu): GELU(approximate='none')
  (dropout): Dropout(p=0.0, inplace=False)
  (flatten): Flatten(start_dim=-2, end_dim=-1)
  (conv1d): Conv1d(7, 2, kernel_size=(20,), stride=(1,))
  (softmax): Softmax(dim=1)
)


torch.Size([4, 2, 1])

In [ ]:
#| export    
class InceptionTime(nn.Module):
    def __init__(self, c_in, n_classes, n_levels=None, dropout=0., bottleneck_channels=32, groups=1, **kwargs):
        super().__init__()
        self.inceptionblock = InceptionBlock(c_in, groups=groups, bottleneck_channels=bottleneck_channels, **kwargs)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten(start_dim=-2)
        self.fc = nn.Linear(bottleneck_channels * 4, n_classes)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, x, return_softmax=False, channel_mask=None):
        """
        x: [bs x n channels x seq len] or x: [bs x n channels x d model x n patches]
        out: [bs x n classes x 1]
        """
        if x.dim() == 4:
            x = x.permute(0,1,3,2)
            x = x.flatten(start_dim=-2)
        x = self.inceptionblock(x)
        x = self.gap(x)
        x = self.flatten(x)
        x = self.fc(x)
        if return_softmax:
            x = self.softmax(x)
        return x
    
class HierarchicalInceptionTime(nn.Module):
    def __init__(self, c_in, n_classes, n_levels, dropout=0.1, bottleneck_channels=32, groups=1, **kwargs):
        super().__init__()
        self.inceptionblock = InceptionBlock(c_in, groups=groups, bottleneck_channels=bottleneck_channels, **kwargs)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten(start_dim=-2)
        self.n_levels = n_levels
        self.fcs = nn.ModuleList()
        for i in range(n_levels):
            layers = [nn.Linear(bottleneck_channels * 4 + i, n_classes)]
            if i < n_levels - 1:  # Don't add dropout after last layer
                layers.append(nn.Dropout(dropout))
            self.fcs.append(nn.Sequential(*layers))
        #self.fcs = nn.ModuleList([nn.Linear(bottleneck_channels * 4 + i, n_classes) for i in range(n_levels)])
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, x, return_softmax=False, channel_mask=None):
        """
        x: [bs x n channels x seq len] or x: [bs x n channels x d model x n patches]
        out: [bs x n classes x 1]
        """
        if x.dim() == 4:
            x = x.permute(0,1,3,2)
            x = x.flatten(start_dim=-2)
        x = self.inceptionblock(x)
        x = self.gap(x)
        x = self.flatten(x)
        pred_levels = []
        for i in range(self.n_levels):
            if i > 0:
                x = torch.cat([x, prev_x], dim=1)
            prev_x = self.fcs[i](x)
            pred_levels.append(prev_x)
        if return_softmax:
            x = self.softmax(x)
        return torch.concat(pred_levels, dim=1)

In [ ]:
#| notest
in_ = torch.randn(4,7,512,10)

mod = InceptionTime(c_in=7, n_classes=1, depth=2, residual=True, bottleneck=True, kernel_size=40)

mod(in_).shape

torch.Size([4, 1])

In [ ]:
#| notest
in_ = torch.randn(4,7,512,10)

mod = HierarchicalInceptionTime(c_in=7, n_classes=1, n_levels=3, bottleneck_channels=32, groups=1, depth=2, residual=True, bottleneck=True, kernel_size=40)

out = mod(in_)

ce_loss = nn.BCEWithLogitsLoss()
print(out.shape)
target = torch.empty(4,3).random_(2)
print(target.shape)
loss = 0
for i in range(3):
    loss += ce_loss(out[:,i], target[:,i])
loss

torch.Size([4, 3])
torch.Size([4, 3])


tensor(2.0436, grad_fn=<AddBackward0>)

In [ ]:
#| notest
in_ = torch.randn(2,7,360)
b = InceptionBlock(in_channels=7, kernel_size=24, bottleneck_channels=8, bottleneck=False, residual=False, depth=3)
b(in_).shape

torch.Size([2, 32, 360])

In [ ]:
#| export
class LinearProbingHead(nn.Module):
    """
    A linear probing head (with optional MLP), assumes that the d_model corresponds to a particular segment of time and will make a prediction per patch per channel, and average the results
    """
    def __init__(self, 
                 c_in, # the number of input channels in the original input
                 predict_every_n_patches, # for a given sequence of length m with frequency f, number of predictions
                 n_classes, # the number of classes to predict (for sleep stage - there are 6)
                 input_size, # the dimension of the transformer model
                 n_layers, # the number of linear layers to use in the prediciton head, with RELU activation and dropout
                 num_patch,
                 shared_embedding=True, # whether or not to have a dense layer per channel or one layer per channel
                 affine=True, # include learnable parameters to weight predictions
                 norm='BatchNorm', # batchnorm or layernorm between linear and convolutional layers
                 act='gelu', # activation function to use between layers, 'gelu' or 'relu'
                 dropout=0. # dropout in between linear layers
                 ):
        super().__init__()
        self.c_in = c_in
        self.n_classes = n_classes
        self.input_size = input_size
        self.n_layers = n_layers
        self.affine = affine
        self.shared_embedding = shared_embedding
        self.predict_every_n_patches = predict_every_n_patches
        self.num_patch = num_patch
        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(1,c_in,1))
            self.affine_bias = nn.Parameter(torch.zeros(1,c_in,1))
        
        n_nuerons = int((self.n_classes - self.input_size)/self.n_layers) if self.n_layers >= 1 else 0

        self.layers = []
        if self.n_layers == 1:
            if not shared_embedding:
                for _ in range(self.c_in): self.layers.append(nn.Linear(input_size, n_classes))
            else:
                self.layers.append(nn.Linear(input_size, n_classes)) # [bs x c_in x n_patches x n_classes]
        else:
            if not shared_embedding:
                for _ in range(self.c_in):
                    channel_layers = []
                    for i in range(self.n_layers-1):
                        channel_layers += [nn.Linear(n_nuerons*i+input_size, input_size+n_nuerons*(i+1)), nn.BatchNorm1d(self.num_patch) if "batch" in norm.lower() else nn.LayerNorm(1), get_activation_fn(act), nn.Dropout(dropout)]
                    channel_layers += [nn.Linear(input_size+n_nuerons*(i+1), self.n_classes)]
                    self.layers.append(nn.Sequential(*channel_layers))
            else:
                for i in range(self.n_layers-1):
                    self.layers.append(nn.Sequential(nn.Linear(n_nuerons*i+input_size, input_size+n_nuerons*(i+1)), Transpose(1,2), nn.BatchNorm2d(c_in) if "batch" in norm.lower() else nn.LayerNorm(c_in), Transpose(1,2), get_activation_fn(act), nn.Dropout(dropout)))
                self.layers.append(nn.Linear(input_size+n_nuerons*(i+1), self.n_classes))
        self.layers = nn.Sequential(*self.layers) # [bs x c_in x n_patches x n_classes], need to average or max over channels
        self.flatten = nn.Flatten(start_dim=-3, end_dim=-2)
        self.avg_pool_kernel_stride = int(self.c_in*self.predict_every_n_patches) if self.predict_every_n_patches > 1 else self.c_in
        self.avg_pool = nn.AvgPool1d(kernel_size=self.avg_pool_kernel_stride, stride=self.avg_pool_kernel_stride)
        self.softmax = nn.LogSoftmax(dim=1)
    
    def forward(self, x, return_softmax=False):
        """
        Make sure to freeze the encoder and remove its pretrain head! 

        in: [bs x n channels x d model x n patches]
        out: [bs x n classes x pred len]
        """
        x = x.permute(0,3,1,2) # [bs x n patches x n channels x d_model]
        if not self.shared_embedding:
            x_out = []
            for i in range(self.c_in): 
                z = self.layers[i](x[:,:,i,:])
                x_out.append(z)
            x = torch.stack(x_out, dim=2) # [bs x n patches x n channels x n_classes]
        else:
            x = self.layers(x) # [bs x n patches x n channels x n_classes]
        if self.affine:
            x = x * self.affine_weight
            x = x + self.affine_bias
        x = self.flatten(x)
        x = x.transpose(1,2)
        x  = self.avg_pool(x)
        if return_softmax:
            x = self.softmax(x)
        return x

In [ ]:
#| notest
m = LinearProbingHead(c_in=7, 
                      input_size = 512, 
                      predict_every_n_patches=3600,
                      n_classes=2,
                      n_layers=2,
                      shared_embedding=False,
                      affine=True,
                      num_patch=3600,
                      dropout=0.1)

x = torch.randn((4,7,512,3600))

m(x, return_softmax=True).shape

torch.Size([4, 2, 1])

In [ ]:
#| export
class LinearProbingHeadBinaryOutcome(nn.Module):
    """
    A linear probing head (with optional MLP), assumes that the d_model corresponds to a particular segment of time and will make a prediction per patch per channel, and average the results
    """
    def __init__(self, 
                 c_in, # the number of input channels in the original input
                 n_patches, # the number of stft or time patches
                 patch_len,
                 d_model, # the dimension of the transformer model
                 norm='BatchNorm', # batchnorm or layernorm between linear and convolutional layers
                 act='gelu', # activation function to use between layers, 'gelu' or 'relu'
                 dropout=0. # dropout in between linear layers
                 ):
        super().__init__()
        self.c_in = c_in
        self.n_classes = 2
        self.n_patches = n_patches
        self.d_model = d_model

        #n_neurons = int((self.n_classes - self.d_model)/self.n_linear_layers) if self.n_linear_layers >= 1 else 0
        self.layer1 = nn.Linear(d_model, patch_len)
        self.norm = nn.BatchNorm2d(c_in) if 'batch' in norm.lower() else nn.LayerNorm([c_in, n_patches, patch_len])
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten(start_dim=-3)
        self.layer2 = nn.Linear(patch_len*n_patches*c_in, self.n_classes) # [d_model x n_patches x c_in]

        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x, return_softmax=False):
        """
        Make sure to freeze the encoder and remove its pretrain head! 

        in: [bs x n channels x d model x n patches]
        out: [bs x n classes x pred len]
        """
        #x = self.relu(self.norm(self.layer1(x)))
        x = x.permute(0,1,3,2)
        x = self.layer1(x)
        x = self.norm(self.relu(x))
        x = self.flatten(x)
        x = self.layer2(x)
        if return_softmax:
            x = self.sigmoid(x)
        return x.unsqueeze(-1)

In [ ]:
#| notest
m = LinearProbingHeadBinaryOutcome(c_in=7, 
                      d_model = 512, 
                      n_patches=3600,
                      dropout=0.1)

x = torch.randn((4,7,512,3600))

m(x, return_softmax=False).shape

torch.Size([4, 2, 1])

In [ ]:
#| export
class MLPProbingHeadBinaryOutcome(nn.Module):
    def __init__(self, 
                 c_in,
                 n_patches,
                 d_model,
                 hidden_dim=1024,
                 dropout=0.1,
                 ):
        super().__init__()
        self.n_classes = 2
        
        # Calculate input dimension for MLP
        self.input_dim = c_in * n_patches * d_model
        
        # MLP classifier
        self.classifier = nn.Sequential(
            nn.Flatten(start_dim=1),  # Flatten everything after batch dimension
            nn.Linear(self.input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),  # Add BatchNorm after linear
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.BatchNorm1d(hidden_dim//2),  # Add BatchNorm after linear
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim//2, self.n_classes)
        )
        
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x, return_softmax=False):
        """
        Args:
            x: Transformer embeddings [bs x n_channels x d_model x n_patches]
            return_softmax: Whether to apply sigmoid activation
        Returns:
            Binary predictions [bs x n_classes x 1]
        """
        x = self.classifier(x)
        
        if return_softmax:
            x = self.sigmoid(x)
            
        return x.unsqueeze(-1)

In [ ]:
#| export
class MLPProbingHead(nn.Module):
    def __init__(self, 
                 c_in,
                 n_patches,
                 d_model,
                 n_classes,
                 hidden_dim=1024,
                 dropout=0.1,
                 pre_norm=False
                 ):
        super().__init__()
        assert n_classes >= 2, "MLPProbingHead must have >= 2 classes. For binary outcomes, set n_classes=2."
        self.n_classes = n_classes
        
        # Calculate input dimension for MLP
        self.input_dim = c_in * n_patches * d_model
        self.pre_norm = pre_norm

        if self.pre_norm:
            self.norm = nn.BatchNorm2d(c_in)
        else:
            self.norm = nn.Identity()
        
        # MLP classifier
        self.classifier = nn.Sequential(
            nn.Flatten(start_dim=1),  # Flatten everything after batch dimension
            nn.Linear(self.input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),  # Add BatchNorm after linear
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.BatchNorm1d(hidden_dim//2),  # Add BatchNorm after linear
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim//2, self.n_classes)
        )
        
        if n_classes == 2:
            self.sigmoid = nn.Sigmoid()
        else:
            self.sigmoid = nn.LogSoftmax(dim=1)
    
    def forward(self, x, return_softmax=False):
        """
        Args:
            x: Transformer embeddings [bs x n_channels x d_model x n_patches]
            return_softmax: Whether to apply sigmoid activation
        Returns:
            Binary predictions [bs x n_classes x 1]
        """
        x = self.norm(x)
        x = self.classifier(x)
        
        if return_softmax:
            x = self.sigmoid(x)
            
        return x.unsqueeze(-1)

In [ ]:
#| notest
m = MLPProbingHead(c_in=1, 
                      d_model=512, 
                      n_patches=10,
                      n_classes=3,
                      hidden_dim=1024,
                      dropout=0.1)

x = torch.randn((4,1,512,10))

m(x, return_softmax=False).shape

torch.Size([4, 3, 1])

In [ ]:
#| export
class LogisticRegression(nn.Module):
    """
    A linear probing head (with optional MLP), assumes that the d_model corresponds to a particular segment of time and will make a prediction per patch per channel, and average the results
    """
    def __init__(self, 
                 c_in, # the number of input channels in the original input
                 n_classes, # the number of classes to predict (for sleep stage - there are 6)
                 input_size, # the dimension of the transformer model
                 n_patches,
                 dropout=0.
                 ):
        super().__init__()
        self.c_in = c_in
        self.n_classes = n_classes
        self.input_size = input_size
        self.n_patches = n_patches
        
        self.linear1 = nn.Linear(c_in * input_size * n_patches, 256)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

        self.flatten = nn.Flatten(start_dim=-3, end_dim=-1)
        
        self.linear2 = nn.Linear(256, n_classes)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, x, return_softmax=False):
        """
        Make sure to freeze the encoder and remove its pretrain head! 

        in: [bs x n channels x d model x n patches]
        out: [bs x n classes x pred len]
        """
        x = x.permute(0,3,1,2) # [bs x n patches x n channels x d_model]
        x = self.flatten(x)
        x = self.linear1(x)
        x = self.dropout(x)
        x = self.relu(x)

        x = self.linear2(x)
        if return_softmax:
            x = self.softmax(x)
        return x.unsqueeze(-1)
        

In [ ]:
#| notest
m = LogisticRegression(c_in=7, 
                      input_size = 512, 
                      n_patches=3600,
                      n_classes=2,
                      )

x = torch.randn((4,7,512,3600))

m(x, return_softmax=False).shape


torch.Size([4, 2])

In [ ]:
#| export
class ProgressiveLogisticRegression(nn.Module):
    """
    Binary logistic regression with progressive dimension reduction
    """
    def __init__(self, 
                 c_in=7,           
                 input_size=512,   
                 n_patches=4800,
                 embedding_reduction_dim=256,
                 patch_reduction_dim=32,
                 dropout=0.
                 ):
        super().__init__()
        self.c_in = c_in
        self.input_size = input_size 
        self.n_patches = n_patches
        self.embedding_reduction_dim = embedding_reduction_dim
        self.patch_reduction_dim = patch_reduction_dim
        
        # First reduce embedding dimension
        self.embed_reduction = nn.Linear(input_size, embedding_reduction_dim)
        
        # Then reduce across patches using average pooling
        self.pool_patches = nn.AdaptiveAvgPool1d(output_size=patch_reduction_dim)
        
        # Calculate reduced dimension
        reduced_dim = c_in * embedding_reduction_dim * patch_reduction_dim  # Much smaller than original
        
        # Final classification layers
        self.classifier = nn.Sequential(
            nn.Linear(reduced_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )
        
        self.sigmoid = nn.LogSigmoid()

    def forward(self, x, return_softmax=False):
        """
        Args:
             in: [bs x n channels x d model x n patches]
            return_softmax: If True, applies sigmoid activation
        Returns:
            Binary predictions [bs x 2 x 1]
        """
        bs = x.size(0)
        
        # Reduce embedding dimension first
        x = x.reshape(bs * self.c_in, self.input_size, self.n_patches)  # Combine batch and channel dims
        x = self.embed_reduction(x.transpose(1, 2))  # [bs*7 x 4800 x 64]

        # Reduce patch dimension
        x = self.pool_patches(x.transpose(1, 2))  # [bs*7 x 64 x 32]
        # Reshape and flatten for final classification
        x = x.reshape(bs, self.c_in * self.embedding_reduction_dim * self.patch_reduction_dim)  # [bs x (7*64*32)]
        
        # Final classification
        x = self.classifier(x)
        
        if return_softmax:
            x = self.sigmoid(x)
            
        return x.unsqueeze(-1)  # [bs x 1 x 1]

In [ ]:
#| notest
m = ProgressiveLogisticRegression(c_in=7, 
                      input_size = 512, 
                      n_patches=3600,
                      embedding_reduction_dim=256,
                      patch_reduction_dim=32,
                      )

x = torch.randn((4,7,512,3600))

m(x, return_softmax=False).shape


torch.Size([4, 2, 1])

In [ ]:
#| export
class AvgPatchLogisticRegression(nn.Module):
    """
    Binary logistic regression with progressive dimension reduction
    """
    def __init__(self, 
                 c_in=7,           
                 input_size=512,   
                 missing_channel_indices=None,
                 dropout=0.
                 ):
        super().__init__()
        self.c_in = c_in
        self.input_size = input_size 
        self.dropout = dropout
        self.missing_channel_indices = missing_channel_indices
        if self.missing_channel_indices is not None:
            # expects : [bs x nvars x d_model x num_patch] 
            self.masked_channel_token_layer = LearnableMaskedChannelTokens(missing_channel_indices=self.missing_channel_indices, d_model=self.input_size) 
        else:
            self.masked_channel_token_layer = nn.Identity()
        # Then reduce across patches using average pooling
        self.pool_patches = nn.AdaptiveAvgPool1d(output_size=1)
                
        # Final classification layers
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(c_in * input_size, 2)
        
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, return_softmax=False, channel_mask=None):
        """
        Args:
             in: [bs x n channels x d model x n patches]
            return_softmax: If True, applies sigmoid activation
        Returns:
            Binary predictions [bs x 2 x 1]
        """
        bs = x.size(0)
        #if channel_mask is not None:
        #    x[channel_mask, :, :] = 0.

        x = self.masked_channel_token_layer(x)
        # Reduce embedding dimension first
        x = x.reshape(bs * self.c_in, self.input_size, -1)  # Combine batch and channel dims [bs*c_in x input_size x n_patches]
        # Reduce patch dimension
        x = self.pool_patches(x)  # [bs*7 x input_size x 1]
        # Reshape and flatten for final classification
        x = x.reshape(bs, self.c_in * self.input_size)  # [bs x (c in*input size)]
        # Final classification
        x = self.dropout(x)
        x = self.classifier(x) # [bs x 2]
        
        if return_softmax:
            x = self.sigmoid(x)
            
        return x.unsqueeze(-1)  # [bs x 2 x 1]

In [ ]:
#| notest
m = AvgPatchLogisticRegression(c_in=7, 
                      input_size = 512, 
                      )

x = torch.randn((4,7,512,3600))
y = torch.randint(0, 2, (4,1))
out = m(x, return_softmax=False)
out.shape


torch.Size([4, 2, 1])

In [ ]:
#| export
class AvgPatchMultiLabel(nn.Module):
    """
    Multilabel logistic regression
    """
    def __init__(self, 
                 n_labels=1,
                 c_in=7,
                 input_size=512,   
                 missing_channel_indices=None,
                 dropout=0.,
                 weibull = False,
                 pre_norm = False
                 ):
        super().__init__()
        self.c_in = c_in
        self.input_size = input_size 
        self.dropout = dropout
        self.missing_channel_indices = missing_channel_indices
        self.weibull = weibull
        self.n_labels = n_labels
        self.pre_norm = pre_norm
        if self.pre_norm:
            self.norm = nn.LayerNorm(input_size)
        if self.missing_channel_indices is not None:
            # expects : [bs x nvars x d_model x num_patch] 
            self.masked_channel_token_layer = LearnableMaskedChannelTokens(missing_channel_indices=self.missing_channel_indices, d_model=self.input_size) 
        else:
            self.masked_channel_token_layer = nn.Identity()
        # Then reduce across patches using average pooling
        self.pool_patches = nn.AdaptiveAvgPool1d(output_size=1)
                
        # Final classification layers
        self.dropout = nn.Dropout(dropout)
        if not self.weibull:
            self.classifier = nn.Linear(c_in * input_size, n_labels)
            self.sigmoid = nn.Sigmoid()
        else:
            # two parameters for shape and scale
            self.classifier = nn.Linear(c_in * input_size, n_labels*2)
            self.softplus = nn.Softplus()
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x, return_softmax=False, channel_mask=None):
        """
        Args:
             in: [bs x n channels x d model x n patches]
            return_softmax: If True, applies sigmoid activation
        Returns:
            Binary predictions [bs x n labels]
        """
        bs = x.size(0)

        x = self.masked_channel_token_layer(x)
        # Reduce embedding dimension first
        # Reduce patch dimension
        if x.is_nested:
            x_list = []
            for x_i in x:
                if self.pre_norm:
                    x_i = x_i.transpose(1, 2)
                    x_i = self.norm(x_i)
                    x_i = x_i.transpose(1, 2)
                x_list.append(x_i.mean(dim=-1))
            x = torch.stack(x_list, dim=0)
        else:
            if self.pre_norm:
                x = x.transpose(2, 3)
                x = self.norm(x)
                x = x.transpose(2, 3)
            x = x.mean(dim=-1)  # [bsx7 x input_size
        # Reshape and flatten for final classification
        x = x.reshape(bs, self.c_in * self.input_size)  # [bs x (c in*input size)]
        # Final classification
        x = self.dropout(x)
        x = self.classifier(x) # [bs x n_labels]
        
        if return_softmax and not self.weibull:
            x = self.sigmoid(x)
        elif self.weibull:
            x = x.reshape(bs, self.n_labels, 2)
            x = self.softplus(x)
            
        return x  # [bs x n_labels]

In [ ]:
#| notest
batch_size = 2
n_vars = 7
max_len = 480
d_model = 384

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, d_model, length) for length in seq_lens]
x_list2 = [torch.randn(n_vars, d_model, length) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)

m = AvgPatchMultiLabel(c_in=n_vars, input_size=d_model, n_labels=100, dropout=0.1)

y = m(x_nested, return_softmax=True)
# from wavefm.loss import mse_loss
# print(y.shape)
# y = y.mean(dim=1)
# print(y.shape)
# mse_loss(y, torch.ones_like(y)).backward()

torch.Size([2, 100])
torch.Size([2])


In [ ]:
#| export
class AvgChannelLogisticRegression(nn.Module):
    """
    Binary logistic regression with channel averaging
    """
    def __init__(self, 
                 n_patches,
                 input_size=512,   
                 dropout=0.
                 ):
        super().__init__()
        self.n_patches = n_patches
        self.input_size = input_size 
        self.dropout = dropout
                
        # Final classification layers
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(n_patches * input_size, 2)
        
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, return_softmax=False, channel_mask=None):
        """
        Args:
             in: [bs x n channels x d model x n patches]
            return_softmax: If True, applies sigmoid activation
        Returns:
            Binary predictions [bs x 2 x 1]
        """
        bs = x.size(0)
        
        # Reduce embedding dimension first
        x = x.mean(dim=1) # [bs x input_size x n_patches]
        # Reshape and flatten for final classification
        x = x.reshape(bs, self.n_patches * self.input_size)  # [bs x (n_patches*input_size)]
        # Final classification
        x = self.dropout(x)
        x = self.classifier(x) # [bs x 2]
        
        if return_softmax:
            x = self.sigmoid(x)
            
        return x.unsqueeze(-1)  # [bs x 2 x 1]

In [ ]:
#| export
class IndividualPatchLinear(nn.Module):
    def __init__(self, 
                 c_in,
                 n_classes,
                 n_patches,
                 mlp_ratio=4,
                 input_size=512,   
                 dropout=0.
                 ):
        super().__init__()
        self.input_size = input_size 
        self.dropout = dropout
        self.c_in = c_in
        self.n_classes = n_classes
        self.num_patches = n_patches
        hidden_features = int(input_size * mlp_ratio)
        self.fc1 = nn.Linear(input_size*c_in, hidden_features)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_features, input_size*c_in)
        self.drop = nn.Dropout(dropout)
        self.fc3 = nn.Linear(input_size*c_in, n_classes)

    def forward(self, x):
        """
        in: [bs x n channels x d model x n patches]
        out: [bs x n classes x pred len]
        """
        bs = x.size(0)
        x = x.permute(0,3,1,2) # [bs x n patches x n channels x d model]
        x = x.flatten(start_dim=2, end_dim=3) # [bs x n channels x d model * n patches]
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        x = self.fc3(x)
        if x.is_nested:
           x = x.to_padded_tensor(padding=0, output_size=(bs, self.num_patches, self.n_classes))
        return x.transpose(1,2)
        

In [ ]:
#| notest
m = IndividualPatchLinear(
                 c_in=7,
                 n_classes=5,
                 n_patches=10,
                 mlp_ratio=4,
                 input_size=512,
                 dropout=0.)

x = torch.randn((4,7,512,10))
m(x).shape


torch.Size([4, 5, 10])

In [ ]:
#| notest
d_model = 256
n_heads = 2
batch_size = 4
n_vars = 7
max_len = 100

m = IndividualPatchLinear(
                 c_in=7,
                 n_classes=5,
                 n_patches=10,
                 mlp_ratio=4,
                 input_size=256,
                 dropout=0.)

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, d_model, length) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)

print(x_nested.shape)
# Forward pass
output = m(x_nested)
output.shape#.transpose(1, 2).flatten(-2).shape
# from wavefm.loss import multilabel_cox_loss

#multilabel_cox_loss(output, torch.tensor([1,0,1,1]), torch.tensor([10,20,23,21]), n_labels=1).backward()

torch.Size([4, 7, 256, j17])


torch.Size([4, 5, 10])

In [ ]:
#| export
class AttentionBinaryClassifier(nn.Module):
    def __init__(self, 
                 c_in, 
                 d_model, 
                 n_heads,
                 n_classes, 
                 num_patch, 
                 attn_dropout=0., 
                 dropout=0.
                 ):
        super().__init__()
        d_k = d_model // n_heads
        d_v = d_model // n_heads
        self.flatten = nn.Flatten(start_dim=-2, end_dim=-1)
        self.proj = nn.Linear(c_in * d_model, d_model)
        self.pe = PositionalEncoding(num_patch=num_patch, d_model=d_model)
        self.attention = MultiheadAttentionCustom(d_model, n_heads, d_k, d_v, res_attention=False, attn_dropout=attn_dropout, proj_dropout=dropout)
        self.fc = nn.Linear(d_model, n_classes)
        self.softmax = nn.LogSoftmax(dim=1)
        
    def forward(self, x, return_softmax=False):
        """
        Make sure to freeze the encoder and remove its pretrain head! 

        in: [bs x n channels x d model x n patches]
        out: [bs x n classes x pred len]
        """
        x = x.permute(0, 3, 1, 2)  # [bs, n_patches, c_in, d_model]
        x = self.flatten(x) # [bs, n_patches, c_in * d_model]
        x = self.proj(x) # [bs, n_patches, d_model]
        x = self.pe(x) # [bs, n_patches, d_model]
        x,_ = self.attention(x) # [bs, n_patches, d_model]
        x = x.mean(dim=1)  # [bs, d_model]
        x = self.fc(x)
        if return_softmax:
            x = self.softmax(x)
        return x.unsqueeze(-1)

In [ ]:
#| notest
m = AttentionBinaryClassifier(c_in=7, 
                      d_model = 512, 
                      n_heads=8,
                      n_classes=2,
                      num_patch=3600
                      )

x = torch.randn((4,7,512,3600))

m(x, return_softmax=False).shape


torch.Size([4, 2, 1])

In [ ]:
#| export
class CrossAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=12,
        qkv_bias=False,
    ):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        #self.q = nn.Linear(dim, dim, bias=qkv_bias)
        #self.kv = nn.Linear(dim, int(dim*2), bias=qkv_bias)
        self.W_Q = nn.Linear(dim, dim, bias=qkv_bias)
        self.W_K = nn.Linear(dim, dim, bias=qkv_bias)
        self.W_V = nn.Linear(dim, dim, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)

    def forward(self, q, x):
        #B, n, C = q.shape
        #q1 = self.q(q).reshape(B, n, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        q = self.W_Q(q).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        #kv = self.kv(x).reshape(B, N, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        #k, v = kv[0], kv[1]  # (batch_size, num_heads, seq_len, feature_dim_per_head)
        k = self.W_K(x).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        v = self.W_V(x).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH], set_priority=True):
            q = F.scaled_dot_product_attention(q, k, v)
        #xattn = (q @ k.transpose(-2, -1)) * self.scale
        #xattn = xattn.softmax(dim=-1)  # (batch_size, num_heads, query_len, seq_len)
        #q = (xattn @ v)
        q = q.transpose(1, 2).flatten(-2)
        q = self.proj(q)
        return q


class CrossAttentionBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=4.,
        qkv_bias=False,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm
    ):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.xattn = CrossAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias)
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = MLP(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer)

    def forward(self, q, x):
        y = self.xattn(q, self.norm1(x))
        q = q + y
        q = q + self.mlp(self.norm2(q))
        return q

class AttentivePooler(nn.Module):
    """ Attentive Pooler """
    def __init__(
        self,
        num_queries=1,
        embed_dim=768,
        num_heads=12,
        mlp_ratio=4.0,
        depth=1,
        norm_layer=nn.LayerNorm,
        init_std=0.02,
        qkv_bias=True,
        complete_block=True,
    ):
        super().__init__()
        self.query_tokens = nn.Parameter(torch.zeros(1, num_queries, embed_dim))

        self.complete_block = complete_block
        if complete_block:
            self.cross_attention_block = CrossAttentionBlock(
                dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                norm_layer=norm_layer)
        else:
            self.cross_attention_block = CrossAttention(
                dim=embed_dim,
                num_heads=num_heads,
                qkv_bias=qkv_bias)

        self.blocks = None
        if depth > 1:
            self.blocks = nn.ModuleList([
                JEPABlock(
                    dim=embed_dim,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    qk_scale=False,
                    norm_layer=norm_layer)
                for i in range(depth-1)])

        self.init_std = init_std
        trunc_normal_(self.query_tokens, std=self.init_std)
        self.apply(self._init_weights)
        self._rescale_blocks()

    def _rescale_blocks(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))

        if self.complete_block:
            rescale(self.cross_attention_block.xattn.proj.weight.data, 1)
            rescale(self.cross_attention_block.mlp.fc2.weight.data, 1)
        else:
            rescale(self.cross_attention_block.proj.weight.data, 1)
        if self.blocks is not None:
            for layer_id, layer in enumerate(self.blocks, 1):
                rescale(layer.attn.proj.weight.data, layer_id + 1)
                rescale(layer.mlp.fc2.weight.data, layer_id + 1)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        q = self.query_tokens.repeat(len(x), 1, 1)
        if x.is_nested:
            q = torch.nested.as_nested_tensor(q, layout=torch.jagged)
        q = self.cross_attention_block(q, x)
        if self.blocks is not None:
            for blk in self.blocks:
                q = blk(q)
        return q

class AttentiveClassifier(nn.Module):
    """ Attentive Classifier """
    def __init__(
        self,
        embed_dim=768,
        num_heads=12,
        mlp_ratio=4.0,
        depth=1,
        norm_layer=nn.LayerNorm,
        init_std=0.02,
        qkv_bias=True,
        num_classes=1000,
        complete_block=True,
        num_queries=1,
        affine=False,
        c_in=7,
        per_channel=False
    ):
        super().__init__()
        if per_channel:
             self.poolers = nn.ModuleList([
                AttentivePooler(
                    num_queries=num_queries,
                    embed_dim=embed_dim,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    depth=depth,
                    norm_layer=norm_layer,
                    init_std=init_std,
                    qkv_bias=qkv_bias,
                    complete_block=complete_block,
                ) for _ in range(c_in)
            ])
        else:
            self.pooler = AttentivePooler(
                num_queries=num_queries,
                embed_dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                depth=depth,
                norm_layer=norm_layer,
                init_std=init_std,
                qkv_bias=qkv_bias,
                complete_block=complete_block,
            )
        self.c_in = c_in
        self.per_channel = per_channel

        if per_channel:
            self.linears = nn.ModuleList(nn.Linear(embed_dim, num_classes) for _ in range(c_in))
        else:
            self.linear = nn.Linear(embed_dim*c_in, num_classes, bias=True)

    def forward(self, x):
        """
        x: [bs * nvars x num_patch x d_model] 
        out: [bs x n_classes]
        """
        #x = x.transpose(2,3) # [bs x nvars x num_patch x d_model]
        B_cin, n_patches, d_model = x.shape
        bs = B_cin // self.c_in
        # if not x.is_nested:
        #     x = torch.reshape(x, (B * nvars, n_patches, d_model)) # u: [bs * nvars x num_patch x d_model]
        # else:
        #     x_list = list(x.unbind())
        #     reshaped_list = []
        #     for x_i in x_list:
        #         # x has shape [cin, var_len, d_model]
        #         # Split into 7 separate tensors
        #         if self.affine:
        #             x_i = x_i * self.affine_weight + self.affine_bias
        #         channel_tensors = list(x_i.unbind(0))
        #         reshaped_list.extend(channel_tensors)
        #     x = torch.nested.as_nested_tensor(reshaped_list, layout=torch.jagged)
        # if x.is_nested:
        #     x_r = []
        #     for x_i in x:
        #         x_r.append(x_i.squeeze(0))
        #     x = torch.stack(x_r, dim=0) # no longer needed nested tensor
        # else:
        #     x = x.squeeze(1)
        if not self.per_channel:
            x = self.pooler(x)#.squeeze(1)
            if x.is_nested:
                #x = unflatten_dim_from_batch(x, self.c_in)
                #print(x.shape)
                x_list = list(x.unbind())
                restored_list = []
                for i in range(0, len(x_list), self.c_in):
                    group = x_list[i:i+self.c_in]
                    stacked = torch.stack(group, dim=0)
                    restored_list.append(stacked.squeeze(1))
                x = torch.stack(restored_list, dim=0)
                x = x.transpose(1,2)
            else:
                x = torch.reshape(x, (-1, self.c_in, d_model)) # z: [bs x nvars x num_patch x d_model]
            x = x.flatten(start_dim=-2)  # [bs x nvars * d_model]
            x = self.linear(x)  # [bs x num_classes]
        else:
            if not x.is_nested:
                x = x.reshape(bs, self.c_in, n_patches, d_model)  # [bs x c_in x num_patch x d_model]
            else:
                x = unflatten_dim_from_batch(x, self.c_in)
            x_out = []
            for i in range(self.c_in):
                for b in range(bs):
                    x_chan = x[b, i, :, :].unsqueeze(0)  # [1 x num_patch x d_model]
                    # if x.is_nested:
                    #     x_chan = torch.nested.as_nested_tensor(x_chan, layout=torch.jagged)
                    pooled = self.poolers[i](x_chan).squeeze(1)  # [1 x d_model]
                    if b == 0:
                        pooled_all = pooled
                    else:
                        pooled_all = torch.cat((pooled_all, pooled), dim=0)
                pred = self.linears[i](pooled_all)  # [bs x num_classes]
                x_out.append(pred)
            x = torch.stack(x_out, dim=1) # [bs x c_in x num_queries x d_model]
            x = x.mean(dim=1) # average over channels

            #     x_chan = x[:, i, :, :]  # [bs x num_patch x d_model]
            #     pooled = self.poolers[i](x_chan).squeeze(1)  # [bs x d_model]
            #     pred = self.linears[i](pooled)  # [bs x num_classes]
            #     x_out.append(pred)
            # x = torch.stack(x_out, dim=1) # [bs x c_in x num_queries x d_model]
            # x = x.mean(dim=1) # average over channels
        return x
    
class AttentiveClassifierNoMelt(nn.Module):
    """ Attentive Classifier """
    def __init__(
        self,
        embed_dim=768,
        num_heads=12,
        mlp_ratio=4.0,
        depth=1,
        norm_layer=nn.LayerNorm,
        init_std=0.02,
        qkv_bias=True,
        num_classes=1000,
        complete_block=True,
        num_queries=1,
        affine=False,
        c_in=7,
    ):
        super().__init__()
        self.pooler = AttentivePooler(
            num_queries=num_queries,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            depth=depth,
            norm_layer=norm_layer,
            init_std=init_std,
            qkv_bias=qkv_bias,
            complete_block=complete_block,
        )

        
        self.linear = nn.Linear(embed_dim, num_classes, bias=True)

    def forward(self, x):
        """
        x: [bs x nvars * num_patch x d_model] 
        out: [bs x n_classes]
        """
        x = self.pooler(x)#.squeeze(1)
        x = x.flatten(start_dim=-2)
        x = self.linear(x)
        return x

In [ ]:
#| notest
#  [bs x nvars x d_model x num_patch] 
x = torch.randn(4 * 7, 50, 256)
cls = AttentiveClassifier(embed_dim=256, c_in=7, num_queries=1, num_heads=2, mlp_ratio=4.0, depth=1, norm_layer=nn.LayerNorm, init_std=0.02, qkv_bias=True, num_classes=1, complete_block=False, per_channel=False)
cls(x).shape

torch.Size([28, 1, 256])


torch.Size([4, 1])

In [ ]:
#| notest
#  [bs x nvars x d_model x num_patch] 

cls_ = AttentiveClassifier(embed_dim=768, num_queries=1, num_heads=8, mlp_ratio=4.0, depth=1, norm_layer=nn.LayerNorm, init_std=0.02, qkv_bias=True, num_classes=5, complete_block=False)
sum(p.numel() for p in cls_.parameters())

2390021

In [ ]:
#| notest
#  [bs x nvars x d_model x num_patch] 
x = torch.randn(4, 7*10, 256)
cls_ = AttentiveClassifierNoMelt(embed_dim=256, num_queries=1, num_heads=2, mlp_ratio=4.0, depth=1, norm_layer=nn.LayerNorm, init_std=0.02, qkv_bias=True, num_classes=1, complete_block=True)
cls_(x).shape

torch.Size([4, 1])

In [ ]:
#| notest
from wavefm.nested import flatten_dim_to_batch
d_model = 256
n_heads = 2
batch_size = 4
n_vars = 7
max_len = 100

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(length, n_vars, d_model) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
x_nested = x_nested.transpose(1,2)  # [bs x nvars x d_model x num_patch]
x_nested = flatten_dim_to_batch(x_nested, 1)

print(x_nested.shape)
# Forward pass
#| notest
#  [bs x nvars x d_model x num_patch] 
cls_ = AttentiveClassifier(embed_dim=256, num_queries=1, num_heads=4, mlp_ratio=4.0, depth=1, norm_layer=nn.LayerNorm, init_std=0.02, qkv_bias=True, num_classes=1, complete_block=True, per_channel=False)
output = cls_(x_nested)
output.shape#.transpose(1, 2).flatten(-2).shape
# from wavefm.loss import multilabel_cox_loss

#multilabel_cox_loss(output, torch.tensor([1,0,1,1]), torch.tensor([10,20,23,21]), n_labels=1).backward()

torch.Size([28, j49, 256])


torch.Size([4, 1])

In [ ]:
#| export
class RNNAttentiveClassifier(nn.Module):
    def __init__(self, 
                 c_in, 
                 rnn_input_size, 
                 rnn_hidden_size, 
                 n_classes,
                 num_heads=12,
                 mlp_ratio=4.0,
                 depth=1,
                 norm_layer=nn.LayerNorm,
                 init_std=0.02,
                 qkv_bias=True,
                 complete_block=True,
                 rnn_module='GRU', 
                 rnn_dropout=0., 
                 num_rnn_layers=1,
                 bidirectional=True, 
                 affine=False, 
                 pre_norm=False,
                 ):
        super().__init__()
        self.c_in = c_in
        self.rnn_input_size = rnn_input_size
        self.rnn_hidden_size = rnn_hidden_size
        self.n_classes = n_classes
        self.num_heads = num_heads
        self.mlp_ratio = mlp_ratio
        self.depth = depth
        self.norm_layer = norm_layer
        self.init_std = init_std
        self.qkv_bias = qkv_bias
        self.complete_block = complete_block
        self.rnn_module = rnn_module
        self.rnn_dropout = rnn_dropout
        self.num_rnn_layers = num_rnn_layers
        self.bidirectional = bidirectional
        self.affine = affine
        self.pre_norm = pre_norm

        if pre_norm:
            self.pre_norm_layer = nn.LayerNorm(self.rnn_input_size)
        else:
            self.pre_norm_layer = nn.Identity()
        
        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(c_in))
            self.affine_bias = nn.Parameter(torch.zeros(c_in))

        self.flatten = nn.Flatten(start_dim=-2, end_dim=-1) # flatten d_model and n channels

        if self.rnn_module.lower() == 'gru':
            self.rnn = nn.GRU(input_size=self.rnn_input_size * self.c_in, hidden_size=self.rnn_hidden_size, num_layers=self.num_rnn_layers, bias=True, batch_first=True, dropout=self.rnn_dropout, bidirectional=self.bidirectional)
        else:
            self.rnn = nn.LSTM(input_size=self.rnn_input_size * self.c_in, hidden_size=self.rnn_hidden_size, num_layers=self.num_rnn_layers, bias=True, batch_first=True, dropout=self.rnn_dropout, bidirectional=self.bidirectional)

        f = 2 if self.bidirectional else 1

        self.classifier = AttentiveClassifier(embed_dim=int(self.rnn_hidden_size * f),
            num_heads=self.num_heads,
            mlp_ratio=self.mlp_ratio,
            depth=self.depth,
            norm_layer=self.norm_layer,
            init_std=self.init_std,
            qkv_bias=self.qkv_bias,
            num_classes=self.n_classes,
            complete_block=self.complete_block,
            num_queries=1)

    def forward(self, x):
        was_nested = x.is_nested
        if was_nested:
           lengths = [i.size(-1) for i in x.unbind()]
           x = x.to_padded_tensor(padding=0)
        x = x.transpose(2,3) # [bs x n channels x n patches x d model]
        x = self.pre_norm_layer(x) # [bs x n channels x n patches x d model]
        x = x.permute(0,2,3,1) # [bs x n patches x d model x n channels]
        if self.affine:
             x = x * self.affine_weight
             x = x + self.affine_bias
        #x = x.mean(dim=-1) # [bs x n patches x d model]
        x = self.flatten(x) # [bs x n patches x n channels * d model]
        if was_nested:
            x = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        x,hidden = self.rnn(x) # [bs x n patches x hidden_size]

        if was_nested:
            # note that we cannot convert back to a nested tensor because of backprop errors through RNN
            x, _ = pad_packed_sequence(x, batch_first=True, padding_value=0, total_length=max(lengths))
            #x_list = [x[i, :length, :].contiguous() for i, length in enumerate(lengths)]
            #x = torch.ne
        # convert to [bs x nvars x d_model x num_patch] 
        x = x.unsqueeze(1).transpose(2,3) # [bs x 1 x n patches x hidden_size]
        x = self.classifier(x)
        return x

In [ ]:
#| notest
x = torch.randn((4,7,256,960))
m = RNNAttentiveClassifier(c_in=7, rnn_input_size=256, rnn_hidden_size=256, n_classes=2, rnn_module='GRU', num_rnn_layers=1, rnn_dropout=0., num_heads=2, mlp_ratio=4.0, depth=1, norm_layer=nn.LayerNorm, init_std=0.02, qkv_bias=True, complete_block=True)
m(x).shape

#| notest
d_model = 256
n_heads = 2
batch_size = 4
n_vars = 7
max_len = 100

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, d_model, length) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)

print(x_nested.shape)
# Forward pass
output = m(x_nested)
output.shape#.transpose(1, 2).flatten(-2).shape
# from wavefm.loss import multilabel_cox_loss

#multilabel_cox_loss(output, torch.tensor([1,0,1,1]), torch.tensor([10,20,23,21]), n_labels=1).backward()

torch.Size([4, 960, 512])
torch.Size([4, 1, 512, 960])
torch.Size([4, 7, 256, j52])
torch.Size([4, 94, 512])
torch.Size([4, 1, 512, 94])


torch.Size([4, 2])

In [ ]:
#| export
class TimeDistributedFeedForward(nn.Module):
    """
    Feed forward head that uses a convolutional layer to reduce channel dimensionality
    Followed by a feedforward network to make 
    """
    def __init__(self, 
                 c_in, # the number of input channels
                 n_classes, # the number of classes to predict (for sleep stage - there are 6)
                 n_patches, # the number of stft or time patches
                 d_model, # the dimension of the transformer model
                 pred_len_seconds, # the sequence multiclass prediction length in seconds
                 n_linear_layers, # the number of linear layers to use in the prediciton head, with RELU activation and dropout
                 conv_kernel_stride_size, # the 1d convolution kernel size and stride, in seconds. If you want every 30 second predicitons, put 30 here.
                 dropout=0. # dropout in between linear layers
                 ):
        super().__init__()
        self.c_in = c_in
        self.n_classes = n_classes
        self.pred_len_seconds = pred_len_seconds
        self.n_linear_layers = n_linear_layers
        self.d_model = d_model
        self.n_patches = n_patches
        self.pool_stride = (d_model*n_patches)//pred_len_seconds
        self.pool_kernel = (d_model*n_patches) - (pred_len_seconds - 1) * self.pool_stride
        #self.pred_len = int(self.pred_len_seconds / (self.hop_length / self.frequency))  # lets predict on a second basis, not on a timepoint basis
        self.flatten = nn.Flatten(start_dim=-2)
        self.layers = []
        n_nuerons = int((self.pred_len_seconds - d_model*n_patches)/self.n_linear_layers) if self.n_linear_layers >= 1 else 0
        if self.n_linear_layers == 1:
            self.layers.append(nn.Linear(d_model*n_patches, self.pred_len_seconds))
        elif self.n_linear_layers == 0:
            self.layers.append(nn.AvgPool1d(kernel_size=self.pool_kernel, stride=self.pool_stride, padding=0))
        else:
            for i in range(self.n_linear_layers-1):
                self.layers.append(nn.Sequential(nn.Linear(n_nuerons*i+d_model*n_patches, d_model*n_patches+n_nuerons*(i+1)), nn.ReLU(), nn.Dropout(dropout)))
            self.layers.append(nn.Linear(d_model*n_patches + n_nuerons*(i+1), self.pred_len_seconds))
        self.layers = nn.Sequential(*self.layers)
        self.conv1d = nn.Conv1d(c_in, n_classes, kernel_size=conv_kernel_stride_size, stride=conv_kernel_stride_size)
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x, return_softmax=False):
        """
        Make sure to freeze the encoder and remove its pretrain head! 

        in: [bs x n channels x d model x n patches]
        out: [bs x n classes x pred len]
        """
        x = x.transpose(2,3)
        x = self.flatten(x)
        x = self.layers(x)
        x = self.conv1d(x)
        if return_softmax:
            x = self.softmax(x)
        return x

In [ ]:
#| export
class ConvGRU1DCell(nn.Module):
    """
    Generate a convolutional GRU cell
    """

    def __init__(self, input_size, hidden_size, kernel_size):
        super().__init__()
        padding = kernel_size // 2
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.reset_gate = nn.Conv2d(input_size + hidden_size, hidden_size, kernel_size, padding=padding)
        self.update_gate = nn.Conv2d(input_size + hidden_size, hidden_size, kernel_size, padding=padding)
        self.out_gate = nn.Conv2d(input_size + hidden_size, hidden_size, kernel_size, padding=padding)

        init.orthogonal_(self.reset_gate.weight)
        init.orthogonal_(self.update_gate.weight)
        init.orthogonal_(self.out_gate.weight)
        init.zeros_(self.reset_gate.bias)
        init.zeros_(self.update_gate.bias)
        init.zeros_(self.out_gate.bias)


    def forward(self, input_, prev_state):

        # get batch and spatial sizes
        batch_size = input_.data.size()[0]
        spatial_size = input_.data.size()[2:]

        # generate empty prev_state, if None is provided
        if prev_state is None:
            state_size = [batch_size, self.hidden_size] + list(spatial_size)
            prev_state = torch.zeros(state_size, device=input_.device)
            
        # data size is [batch, channel, height, width]
        stacked_inputs = torch.cat([input_, prev_state], dim=1)
        update = F.sigmoid(self.update_gate(stacked_inputs))
        reset = F.sigmoid(self.reset_gate(stacked_inputs))
        out_inputs = F.tanh(self.out_gate(torch.cat([input_, prev_state * reset], dim=1)))
        new_state = prev_state * (1 - update) + out_inputs * update

        return new_state


class ConvGRU1D(nn.Module):
    def __init__(self, input_size, hidden_sizes, kernel_sizes, n_layers):
        '''
        Generates a multi-layer convolutional GRU.
        Preserves spatial dimensions across cells, only altering depth.

        Parameters
        ----------
        input_size : integer. depth dimension of input tensors.
        hidden_sizes : integer or list. depth dimensions of hidden state.
            if integer, the same hidden size is used for all cells.
        kernel_sizes : integer or list. sizes of Conv2d gate kernels.
            if integer, the same kernel size is used for all cells.
        n_layers : integer. number of chained `ConvGRUCell`.
        '''

        super(ConvGRU1D, self).__init__()

        self.input_size = input_size
        if type(hidden_sizes) != list:
            self.hidden_sizes = [hidden_sizes]*n_layers
        else:
            assert len(hidden_sizes) == n_layers, '`hidden_sizes` must have the same length as n_layers'
            self.hidden_sizes = hidden_sizes
        if type(kernel_sizes) != list:
            self.kernel_sizes = [kernel_sizes]*n_layers
        else:
            assert len(kernel_sizes) == n_layers, '`kernel_sizes` must have the same length as n_layers'
            self.kernel_sizes = kernel_sizes

        self.n_layers = n_layers
        cells = nn.ModuleList()
        for i in range(self.n_layers):
            if i == 0:
                input_dim = self.input_size
            else:
                input_dim = self.hidden_sizes[i-1]

            cell = ConvGRU1DCell(input_dim, self.hidden_sizes[i], self.kernel_sizes[i])
            name = 'ConvGRU1DCell_' + str(i).zfill(2)

            setattr(self, name, cell)
            cells.append(getattr(self, name))
        self.cells = cells

    def forward(self, x, hidden=None):
        '''
        x: [batch, n_channels, dim_model, n_patches]
        hidden: [batch, n_channels, dim_model, n_patches]

        Returns
        -------
        upd_cell_hidden: [batch, n_channels, dim_model, n_patches]
        '''
        if not hidden:
            hidden = [None]*self.n_layers

        input_ = x

        upd_hidden = []

        for layer_idx in range(self.n_layers):
            cell = self.cells[layer_idx]
            cell_hidden = hidden[layer_idx]

            # pass through layer
            upd_cell_hidden = cell(input_, cell_hidden)
            #upd_hidden.append(upd_cell_hidden)
            # update input_ to the last updated hidden layer for next pass
            input_ = upd_cell_hidden

        # retain tensors in list to allow different hidden sizes
        return upd_cell_hidden

class ConvBiGRU(nn.Module):
    # Constructor
    def __init__(self, c_in, d_model, hidden_sizes, kernel_sizes, n_layers, predict_every_n_patches, n_classes):
        super(ConvBiGRU, self).__init__()
        self.c_in = c_in
        self.flatten = nn.Flatten(start_dim=-3, end_dim=-2)
        self.predict_every_n_patches = predict_every_n_patches
        self.n_classes = n_classes
        self.d_model = d_model
        self.hidden_sizes = hidden_sizes
        self.forward_net = ConvGRU1D(c_in, hidden_sizes, kernel_sizes, n_layers)
        self.reverse_net = ConvGRU1D(c_in, hidden_sizes, kernel_sizes, n_layers)

        #self.flatten = nn.Flatten(start_dim=-3, end_dim=-2)
        self.linear = nn.Linear(d_model, n_classes) # [bs x c_in x n_patches x n_classes]
        self.pool_kernel_size = int(self.hidden_sizes*2*self.predict_every_n_patches) if self.predict_every_n_patches > 1 else self.hidden_sizes*2
        self.pool_layer = nn.AvgPool1d(kernel_size=self.pool_kernel_size, stride=self.pool_kernel_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, x, return_softmax=False):
        """
        x: [batch, n_channels, dim_model, n_patches]
        """
        x = x.permute(0, 1, 3, 2)
        #x = self.flatten(x)
        x_rev = torch.flip(x, dims=[-1])
        y_out_fwd = self.forward_net(x)
        y_out_rev = self.reverse_net(x_rev)
        
        y_out_rev = torch.flip(y_out_rev, dims=[-1])
        #reversed_idx = list(reversed(range(y_out_rev.shape[1])))
        #y_out_rev = y_out_rev[:, reversed_idx, ...] # reverse temporal outputs.
        ycat = torch.cat((y_out_fwd, y_out_rev), dim=1)
        ycat = self.linear(ycat)
        ycat = self.flatten(ycat)
        ycat = ycat.transpose(1,2)
        ycat = self.pool_layer(ycat)
        if return_softmax:
            ycat = self.softmax(ycat)
        return ycat

NameError: name 'nn' is not defined

In [ ]:
#| notest
x = torch.randn((4,7,512,3600))

convgru = ConvBiGRU(c_in = 7, d_model=512, hidden_sizes=32, kernel_sizes=3, n_layers=1,  predict_every_n_patches=3600, n_classes=2)

out = convgru(x)
out.shape

In [ ]:
#| export
class LRU(nn.Module):
    def __init__(self,in_features,out_features,state_features, rmin=0, rmax=1, max_phase=6.283):
        """
        Derived from https://github.com/Gothos/LRU-pytorch/blob/main/LRU_pytorch/LRU.py
        and https://github.com/sustcsonglin/flash-linear-rnn/blob/master/linear_rnn/layers/lru.py


        """
        super().__init__()
        self.out_features = out_features
        self.D = nn.Parameter(torch.randn([out_features,in_features])/math.sqrt(in_features))
        
        u1 = torch.rand(state_features)
        u2 = torch.rand(state_features)
        self.nu_log = nn.Parameter(torch.log(-0.5*torch.log(u1*(rmax+rmin)*(rmax-rmin) + rmin**2)))
        self.theta_log = nn.Parameter(torch.log(max_phase*u2))
        self.Lambda_mod = torch.exp(-torch.exp(self.nu_log))
        self.gamma_log = nn.Parameter(torch.log(torch.sqrt(torch.ones_like(Lambda_mod)-torch.square(Lambda_mod))))

        B_re = torch.randn([state_features, in_features])/math.sqrt(2*in_features)
        B_im = torch.randn([state_features, in_features])/math.sqrt(2*in_features)
        self.B = nn.Parameter(torch.complex(B_re, B_im))
        C_re = torch.randn([out_features, state_features])/math.sqrt(state_features)
        C_im = torch.randn([out_features, state_features])/math.sqrt(state_features)
        self.C = nn.Parameter(torch.complex(C_re, C_im))
        self.state = torch.complex(torch.zeros(state_features), torch.zeros(state_features))

    # def initializer(self):
    #     #https://arxiv.org/pdf/2303.06349.pdf Sect.3.2.2
    #     r_min, r_max = 0.9, 0.999 # best result was found with these values
    #     u1 = np.random.random(self.d_model)
    #     u2 = np.random.random(self.d_model)
    #     nu_log = np.log(
    #         -0.5 * np.log(u1 * (r_max**2 - r_min**2) + r_min**2)
    #     )
    #     theta_log = np.log(u2 * np.pi * 2) # max phase is 2pi
    #     gamma_log = np.log(np.sqrt(1 - np.exp(-np.exp(nu_log))**2))
        
    #     return torch.Tensor(nu_log), torch.Tensor(theta_log), torch.Tensor(gamma_log)

    def forward(self, input, state=None):
        self.state = self.state.to(self.B.device) if state==None else state
        Lambda_re = self.Lambda_mod*torch.cos(torch.exp(self.theta_log))
        Lambda_im = self.Lambda_mod*torch.sin(torch.exp(self.theta_log))
        Lambda = torch.complex(Lambda_re, Lambda_im)
        Lambda = Lambda.to(self.state.device)
        gammas = torch.exp(self.gamma_log).unsqueeze(-1).to(self.B.device)
        gammas=gammas.to(self.state.device)
        output=torch.empty([i for i in input.shape[:-1]] +[self.out_features],device=self.B.device)
        #Handle input of (Batches,Seq_length, Input size)
        if input.dim()==3:
            for i,batch in enumerate(input):
                out_seq=torch.empty(input.shape[1],self.out_features)
                for j,step in enumerate(batch):
                    self.state=(Lambda*self.state + gammas* self.B@step.to(dtype= self.B.dtype))
                    out_step= (self.C@self.state).real + self.D@step
                    out_seq[j]=out_step
                self.state=torch.complex(torch.zeros_like(self.state.real),torch.zeros_like(self.state.real))
                output[i]=out_seq
        #Handle input of (Seq_length, Input size)
        if input.dim()==2:
            for i,step in enumerate(input):
                self.state=(Lambda*self.state + gammas* self.B@step.to(dtype= self.B.dtype))
                out_step= (self.C@self.state).real + self.D@step
                output[i]=out_step
            self.state=torch.complex(torch.zeros_like(self.state.real),torch.zeros_like(self.state.real))
        return output

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()